# 📊 YouTube Trending Videos - Data Preparation

**Project:** YZV475E Data Visualization Term Project  
**Team:** EnAi (Muhammed Abdullah Özdemir, Hikmet Gültekin)

---

## Notebook Overview
1. Load and merge all country CSVs
2. Parse category mappings from JSON files
3. Data cleaning and transformation
4. Initial data exploration
5. Export cleaned data for Tableau

In [ ]:
# Core imports
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Libraries imported successfully")

In [ ]:
# Path configuration - BU KISMI KENDİ YAPISINA GÖRE DÜZENLE
BASE_PATH = Path("../input/youtube-trending-video-dataset")
OUTPUT_PATH = Path("../output")
OUTPUT_PATH.mkdir(exist_ok=True)
(OUTPUT_PATH / "aggregated").mkdir(exist_ok=True)
(OUTPUT_PATH / "figures").mkdir(exist_ok=True)

# Country codes in dataset
COUNTRIES = ['BR', 'CA', 'DE', 'FR', 'GB', 'IN', 'JP', 'KR', 'MX', 'RU', 'US']

print(f"📂 Input path: {BASE_PATH}")
print(f"📂 Output path: {OUTPUT_PATH}")
print(f"🌍 Countries to process: {COUNTRIES}")

---
## 1. Load and Merge Country Data

In [ ]:
def load_country_data(base_path, countries):
    """
    Load and merge all country CSV files.
    Adds 'country' column to identify source.
    """
    dfs = []
    
    for country in countries:
        file_path = base_path / f"{country}_youtube_trending_data.csv"
        
        if file_path.exists():
            df = pd.read_csv(file_path)
            df['country'] = country
            dfs.append(df)
            print(f"✅ {country}: {len(df):,} rows loaded")
        else:
            print(f"❌ {country}: File not found")
    
    merged_df = pd.concat(dfs, ignore_index=True)
    print(f"\n📊 Total rows after merge: {len(merged_df):,}")
    
    return merged_df

# Load all data
df = load_country_data(BASE_PATH, COUNTRIES)

In [ ]:
# Quick look at the data structure
print("📋 Dataset Shape:", df.shape)
print("\n📋 Column Names:")
print(df.columns.tolist())
print("\n📋 Data Types:")
print(df.dtypes)

In [ ]:
# Sample rows
df.head(3)

---
## 2. Load Category Mappings

In [ ]:
def load_category_mappings(base_path, countries):
    """
    Load category ID to name mappings from JSON files.
    Creates a unified mapping (some categories are universal).
    """
    all_categories = {}
    
    for country in countries:
        json_path = base_path / f"{country}_category_id.json"
        
        if json_path.exists():
            with open(json_path, 'r') as f:
                data = json.load(f)
            
            for item in data.get('items', []):
                cat_id = int(item['id'])
                cat_name = item['snippet']['title']
                all_categories[cat_id] = cat_name
    
    print(f"📚 Loaded {len(all_categories)} unique categories")
    return all_categories

category_mapping = load_category_mappings(BASE_PATH, COUNTRIES)
print("\n📚 Category Mapping:")
for cat_id, cat_name in sorted(category_mapping.items()):
    print(f"   {cat_id}: {cat_name}")

---
## 3. Data Cleaning & Transformation

In [ ]:
# 3.1 Missing Values Analysis
print("🔍 Missing Values Analysis:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Percentage': missing_pct})
print(missing_df[missing_df['Missing'] > 0])

if missing_df['Missing'].sum() == 0:
    print("✅ No missing values found!")

In [ ]:
# 3.2 Add category names
df['category_name'] = df['categoryId'].map(category_mapping)

# Check unmapped categories
unmapped = df[df['category_name'].isna()]['categoryId'].unique()
if len(unmapped) > 0:
    print(f"⚠️ Unmapped category IDs: {unmapped}")
    df['category_name'] = df['category_name'].fillna('Unknown')
else:
    print("✅ All categories mapped successfully")

In [ ]:
# 3.3 DateTime conversions
df['publishedAt'] = pd.to_datetime(df['publishedAt'], errors='coerce')
df['trending_date'] = pd.to_datetime(df['trending_date'], errors='coerce')

# Extract useful time components
df['publish_year'] = df['publishedAt'].dt.year
df['publish_month'] = df['publishedAt'].dt.month
df['publish_year_month'] = df['publishedAt'].dt.to_period('M').astype(str)
df['publish_day_of_week'] = df['publishedAt'].dt.day_name()
df['publish_hour'] = df['publishedAt'].dt.hour

df['trending_year'] = df['trending_date'].dt.year
df['trending_month'] = df['trending_date'].dt.month
df['trending_year_month'] = df['trending_date'].dt.to_period('M').astype(str)

# Days to trend (how long after publish did it trend?)
df['days_to_trend'] = (df['trending_date'] - df['publishedAt']).dt.days

print("✅ DateTime features created")
print(f"\n📅 Date Range: {df['trending_date'].min()} to {df['trending_date'].max()}")

In [ ]:
# 3.4 Engagement metrics
# Like ratio (likes / views)
df['like_ratio'] = (df['likes'] / df['view_count']).replace([np.inf, -np.inf], np.nan)

# Comment ratio (comments / views)  
df['comment_ratio'] = (df['comment_count'] / df['view_count']).replace([np.inf, -np.inf], np.nan)

# Engagement score (likes + comments) / views
df['engagement_rate'] = ((df['likes'] + df['comment_count']) / df['view_count']).replace([np.inf, -np.inf], np.nan)

print("✅ Engagement metrics calculated")

In [ ]:
# 3.5 Tag processing
def count_tags(tag_string):
    """Count number of tags in a video"""
    if pd.isna(tag_string) or tag_string == '[None]':
        return 0
    return len(str(tag_string).split('|'))

df['tag_count'] = df['tags'].apply(count_tags)

print("✅ Tag count calculated")
print(f"📊 Average tags per video: {df['tag_count'].mean():.1f}")

In [ ]:
# 3.6 Title length (might correlate with engagement)
df['title_length'] = df['title'].str.len()

print("✅ Title length calculated")

In [ ]:
# 3.7 Country full names for better visualization
country_names = {
    'BR': 'Brazil',
    'CA': 'Canada', 
    'DE': 'Germany',
    'FR': 'France',
    'GB': 'United Kingdom',
    'IN': 'India',
    'JP': 'Japan',
    'KR': 'South Korea',
    'MX': 'Mexico',
    'RU': 'Russia',
    'US': 'United States'
}

df['country_name'] = df['country'].map(country_names)

print("✅ Country names added")

---
## 4. Data Quality Check

In [ ]:
print("="*60)
print("📊 CLEANED DATA SUMMARY")
print("="*60)
print(f"\n📏 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\n📅 Time Period: {df['trending_date'].min().strftime('%Y-%m-%d')} to {df['trending_date'].max().strftime('%Y-%m-%d')}")
print(f"\n🌍 Countries: {df['country'].nunique()} ({', '.join(sorted(df['country'].unique()))})")
print(f"\n🎬 Unique Videos: {df['video_id'].nunique():,}")
print(f"\n📺 Unique Channels: {df['channelTitle'].nunique():,}")
print(f"\n🏷️ Categories: {df['category_name'].nunique()}")

In [ ]:
# Rows per country
print("\n🌍 Rows per Country:")
country_counts = df['country'].value_counts()
for country, count in country_counts.items():
    print(f"   {country_names[country]:20} ({country}): {count:>10,} rows")

In [ ]:
# Videos per category
print("\n🏷️ Videos per Category:")
category_counts = df['category_name'].value_counts()
for cat, count in category_counts.head(15).items():
    print(f"   {cat:25}: {count:>10,} ({count/len(df)*100:.1f}%)")

In [ ]:
# Numeric columns summary
print("\n📈 Numeric Columns Summary:")
numeric_cols = ['view_count', 'likes', 'dislikes', 'comment_count', 'tag_count', 'days_to_trend']
df[numeric_cols].describe().round(2)

In [ ]:
# Yearly distribution
print("\n📅 Yearly Distribution:")
yearly = df['trending_year'].value_counts().sort_index()
for year, count in yearly.items():
    print(f"   {int(year)}: {count:>10,} ({count/len(df)*100:.1f}%)")

In [ ]:
# Check for duplicates (same video trending multiple times is NORMAL)
print("\n🔄 Duplicate Analysis:")
print(f"   Total rows: {len(df):,}")
print(f"   Unique video_id: {df['video_id'].nunique():,}")
print(f"   Avg times a video trends: {len(df) / df['video_id'].nunique():.1f}")

# Same video in same country on same day = true duplicate
true_dupes = df.duplicated(subset=['video_id', 'country', 'trending_date']).sum()
print(f"   True duplicates (same video/country/date): {true_dupes:,}")

---
## 5. Final Column List

In [ ]:
print("📋 Final Columns in Dataset:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i:2}. {col}")

In [ ]:
# Final sample
df.head()

---
## 6. Export Cleaned Data

Tableau için veriyi export ediyoruz. 2.9M row çok büyük olabilir, gerekirse sampling yapacağız.

In [ ]:
# Option 1: Full data export (if Tableau can handle it)
# df.to_csv(OUTPUT_PATH / 'cleaned_full_data.csv', index=False)

# Option 2: Sampled data for better Tableau performance
# We'll keep all unique videos but sample their trending appearances

print("💾 Preparing data for export...")
print(f"   Current size: {len(df):,} rows")

# Remove true duplicates first
df_clean = df.drop_duplicates(subset=['video_id', 'country', 'trending_date'])
print(f"   After removing duplicates: {len(df_clean):,} rows")

In [ ]:
# Export full cleaned data
export_path = OUTPUT_PATH / 'youtube_trending_cleaned.csv'
df_clean.to_csv(export_path, index=False)
file_size = os.path.getsize(export_path) / (1024*1024)  # MB
print(f"\n✅ Exported to: {export_path}")
print(f"📦 File size: {file_size:.1f} MB")

---
## 7. Pre-aggregated Data for Tableau Performance

Büyük veri setlerinde Tableau yavaşlayabilir. Bazı analizler için önceden aggregate edilmiş veriler hazırlıyoruz.

In [ ]:
# 7.1 Monthly trends by country
monthly_trends = df_clean.groupby(['trending_year_month', 'country', 'country_name']).agg({
    'video_id': 'count',
    'view_count': 'sum',
    'likes': 'sum',
    'comment_count': 'sum'
}).reset_index()
monthly_trends.columns = ['year_month', 'country', 'country_name', 'video_count', 'total_views', 'total_likes', 'total_comments']

monthly_trends.to_csv(OUTPUT_PATH / 'aggregated' / 'monthly_trends.csv', index=False)
print("✅ monthly_trends.csv exported")

In [ ]:
# 7.2 Country × Category matrix
country_category = df_clean.groupby(['country', 'country_name', 'category_name']).agg({
    'video_id': 'count',
    'view_count': 'mean',
    'likes': 'mean',
    'engagement_rate': 'mean'
}).reset_index()
country_category.columns = ['country', 'country_name', 'category', 'video_count', 'avg_views', 'avg_likes', 'avg_engagement']

country_category.to_csv(OUTPUT_PATH / 'aggregated' / 'country_category_matrix.csv', index=False)
print("✅ country_category_matrix.csv exported")

In [ ]:
# 7.3 Top tags extraction
def extract_all_tags(df):
    """Extract all tags with their frequencies"""
    all_tags = []
    
    for _, row in df.iterrows():
        if pd.notna(row['tags']) and row['tags'] != '[None]':
            tags = str(row['tags']).split('|')
            for tag in tags:
                tag = tag.strip().lower()
                if tag and len(tag) > 1:
                    all_tags.append({
                        'tag': tag,
                        'country': row['country'],
                        'category': row['category_name']
                    })
    
    return pd.DataFrame(all_tags)

# This might take a while for full data, sample if needed
print("⏳ Extracting tags (this may take a few minutes)...")
sample_for_tags = df_clean.sample(n=min(100000, len(df_clean)), random_state=42)
tags_df = extract_all_tags(sample_for_tags)

# Aggregate tag frequencies
tag_freq = tags_df.groupby('tag').size().reset_index(name='count')
tag_freq = tag_freq.sort_values('count', ascending=False).head(500)

tag_freq.to_csv(OUTPUT_PATH / 'aggregated' / 'top_tags.csv', index=False)
print(f"✅ top_tags.csv exported ({len(tag_freq)} tags)")

In [ ]:
# 7.4 Category performance summary
category_summary = df_clean.groupby('category_name').agg({
    'video_id': 'count',
    'view_count': ['mean', 'median', 'sum'],
    'likes': ['mean', 'median'],
    'comment_count': ['mean', 'median'],
    'engagement_rate': 'mean',
    'days_to_trend': 'mean'
}).round(2)

category_summary.columns = ['_'.join(col).strip() for col in category_summary.columns]
category_summary = category_summary.reset_index()

category_summary.to_csv(OUTPUT_PATH / 'aggregated' / 'category_summary.csv', index=False)
print("✅ category_summary.csv exported")

In [ ]:
# 7.5 Hourly publish patterns
hourly_patterns = df_clean.groupby(['publish_hour', 'country']).agg({
    'video_id': 'count',
    'view_count': 'mean'
}).reset_index()
hourly_patterns.columns = ['hour', 'country', 'video_count', 'avg_views']

hourly_patterns.to_csv(OUTPUT_PATH / 'aggregated' / 'hourly_patterns.csv', index=False)
print("✅ hourly_patterns.csv exported")

---
## 8. Export Summary

In [ ]:
print("="*60)
print("📦 EXPORT SUMMARY")
print("="*60)
print("\n📁 Main data:")
print(f"   • youtube_trending_cleaned.csv")

print("\n📁 Aggregated data (for better Tableau performance):")
print(f"   • aggregated/monthly_trends.csv")
print(f"   • aggregated/country_category_matrix.csv")
print(f"   • aggregated/top_tags.csv")
print(f"   • aggregated/category_summary.csv")
print(f"   • aggregated/hourly_patterns.csv")

print("\n✅ Data is ready for Tableau!")

---
## Next Steps

1. Run `02_eda.ipynb` for detailed exploratory analysis
2. Open Tableau and connect to `youtube_trending_cleaned.csv`
3. Use aggregated CSVs for specific visualizations that need better performance